In [12]:
import torch
import torch.nn as nn
import torch.optim as optimizer

import torchvision # a library which is specifically for computer vision
# it includes:
# popular datasets such as CIFAR10, MNIST
# pre trained CNNs
# Utilities for image transformation 

In [2]:
from torchvision.datasets import CIFAR10

In [3]:
# Datasets & Dataloaders:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image => scale (0,1) => Normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./Data", train=True, download=True, transform= transform)
testset = CIFAR10(root="./Data", train=False, download=True, transform= transform)

In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset, batch_size=64, shuffle= True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [10]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )


## the input size is 32X32X3 so n=32, filter=3, stride=1, padding val=1
## to calculate output =    n-f+2p  
#                          --------  +1    = 32
#                              s
# so output will be 32X32
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)   # assuming 10 output classes
        )

    def forward(self, x):   # fixed 'selc' → 'self'
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)   # flattening
        x = self.fc_layers(x)
        return x


In [13]:
model = CNN()

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optimizer.Adam(model.parameters())

### Training the CNN:

In [ ]:
epochs = 10

for epoch in range(epochs):
    training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output= model.forward(images)
        loss =criterion(output, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss = {training_loss/len(trainloader)}")

In [ ]:
# Evalueating our CNN:

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {100 * correct_labels / total_labels}%") 